# Communicate with a vLLM-served Qwen3-VL-235B-A22B model

Sanity-check notebook: pick a sample, overlay bbox annotations on the video, save to disk, then send to the local vLLM server and print the model's answer.

**Prerequisites:** launch the server with `bash bash_scripts/vllm_serving/qwen3vl235b.sh`.

---
# Section 1: V-STaR Dataset

## 1. Load the V-STaR test set

In [ ]:
import json
import datasets

dataset = datasets.load_dataset("json", data_files="/projects/torresani-lab/ajay/datasets/V-STaR/V_STaR_test_with_qid.json", split="train")
print(len(dataset))

## 2. Pick a sample and decode its video

In [ ]:
import os
import sys
import importlib

import numpy as np
import torch
from qwen_vl_utils import fetch_video

sys.path.append("/projects/torresani-lab/ajay/visual-grounded-vlm/")
import src.utils.annotation_utils
importlib.reload(src.utils.annotation_utils)
from src.utils.annotation_utils import load_mask_annotations, annotate_frames

# Pick a sample and decode at fixed token budget
idx = 3
sample = dataset[idx]
print(sample['vid'], sample['qid'])

video_path = os.path.join("/projects/torresani-lab/ajay/datasets/V-STaR/videos", sample['vid'] + ".mp4")
video_input = fetch_video({
    "video": video_path,
    "min_pixels": 128 * 28 * 28,
    "max_pixels": 128 * 28 * 28,
    "fps": 1,
})
print("video_input:", video_input.shape)

## 3. Overlay annotations on the decoded frames

In [ ]:
annotation_root = "/scratch/rajendrakumar.a/visual-grounded-vlm/annotations/vstar/uniform_sample_cap_768/2026-05-19_19-14-38"
annotation_type = "bbox"  # "bbox" | "mask" | "outline"

# Options
annotate_bbox = True
annotate_frame_nums = True

annotated_input = video_input.clone()

if annotate_bbox:
    annotation_path = os.path.join(annotation_root, "masks", f"{sample['qid']}.json")
    record = load_mask_annotations(annotation_path)
    annotated_input = annotate_frames(annotated_input, record["masks_np"], annotation_type=annotation_type)
    annotated_input = torch.tensor(np.array(annotated_input)).permute(0, 3, 1, 2).to(torch.float32)

# Uniformly sample 16 frames
indices = np.linspace(0, annotated_input.shape[0] - 1, 16, dtype=int)
annotated_input = annotated_input[indices]

if annotate_frame_nums:
    import cv2

    def annotate_frame_numbers(frames: torch.Tensor, scale: float = 2.0, thickness: int = 3) -> torch.Tensor:
        annotated = []
        for i, frame in enumerate(frames):
            arr = cv2.cvtColor(frame.permute(1, 2, 0).numpy().astype(np.uint8), cv2.COLOR_RGB2BGR)
            h, w = arr.shape[:2]
            text = str(i)
            font = cv2.FONT_HERSHEY_SIMPLEX
            (tw, th), baseline = cv2.getTextSize(text, font, scale, thickness)
            x, y = w - tw - 8, h - baseline - 4
            cv2.putText(arr, text, (x, y), font, scale, (0, 0, 255), thickness, cv2.LINE_AA)
            annotated.append(torch.from_numpy(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)).permute(2, 0, 1))
        return torch.stack(annotated)

    annotated_input = annotate_frame_numbers(annotated_input)

print("annotated_input:", annotated_input.shape)

## 4. Write the annotated video to disk

In [ ]:
from torchcodec.encoders import VideoEncoder

dump_dir = "/projects/torresani-lab/ajay/visual-grounded-vlm/scripts/vllm_serving/dump"
os.makedirs(dump_dir, exist_ok=True)
output_video_path = os.path.join(dump_dir, f"ann_{sample['qid']}.mp4")
VideoEncoder(annotated_input.to(torch.uint8), frame_rate=4).to_file(output_video_path)
print("wrote:", output_video_path)

## 5. Preview a random annotated frame

In [ ]:
import matplotlib.pyplot as plt

random_idx = np.random.randint(0, annotated_input.shape[0])
any_frame = annotated_input[random_idx].permute(1, 2, 0).numpy().astype(np.uint8)
plt.imshow(any_frame)
plt.title(f"Annotated frame {random_idx} — qid: {sample['qid']}")
plt.axis('off')
plt.show()

## 6. Build the chat request and send to vLLM

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
model = "Qwen/Qwen3-VL-235B-A22B-Instruct"

SYSTEM_PROMPT_PATH = "/projects/torresani-lab/ajay/visual-grounded-vlm/prompts/train/openo3_red_bbox_frame_num_v2.md"
with open(SYSTEM_PROMPT_PATH, "r") as f:
    SYSTEM_PROMPT = f.read()

n_frames = annotated_input.shape[0]
frame_prompt = "".join(f"Frame {i}: <image>\n" for i in range(n_frames))

message = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": frame_prompt + sample['question']},
            {
                "type": "video_url",
                "video_url": {"url": "file://" + output_video_path},
                "VIDEO_MIN_PIXELS": 128 * 28 * 28,
                "VIDEO_MAX_PIXELS": 128 * 28 * 28,
                "fps": 1,
            },
        ],
    },
]

client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)

In [ ]:
chat_response = client.chat.completions.create(
    model=model,
    messages=message,
    max_tokens=768,
    seed=42,
    temperature=0,
)
response_text = chat_response.choices[0].message.content
print(response_text)

In [ ]:
print(f"Question: {sample['question']}")
print(f"Ground truth answer: {sample['answer']}")

---
# Section 2: Open-o3-Video Dataset (STGR)

## OO3-1. Load dataset and track manifest

In [ ]:
import sys
sys.path.append("/projects/torresani-lab/ajay/visual-grounded-vlm/")

import importlib
import json
import os

import cv2
import numpy as np
import torch
from datasets import load_dataset
from qwen_vl_utils import fetch_video

import src.utils.openo3_utils as openo3_utils
importlib.reload(openo3_utils)
from src.utils.openo3_utils import (
    get_video_path, splice_keyframes,
    load_track_manifest, load_track_masks, masks_to_bboxes,
)

# --- config ---
OO3_JSON    = "/projects/torresani-lab/ajay/datasets/Open-o3-Video/json_data/STGR-SFT.json"
RESULTS_DIR = "/scratch/rajendrakumar.a/visual-grounded-vlm/annotations/openo3video/uniform_sample_1fps_orig_res_track"

dataset_oo3 = load_dataset("json", data_files=OO3_JSON, split="train")
str_dataset = [s for s in dataset_oo3 if s["task"] == "temporal-spatial free-form QA"]
run_meta, samples = load_track_manifest(RESULTS_DIR)

print(f"Loaded {len(str_dataset)} STGR samples")
print(f"Track manifest: {len(samples)} entries")

## OO3-2. Pick a sample and reconstruct the annotated video

In [ ]:
# --- config ---
SAMPLE_IDX    = 0
VIDEO_FPS     = run_meta["config"]["video"]["fps"]
TGT_FRAMES    = 32
ANNOTATE_BBOX = True

sample_oo3 = str_dataset[SAMPLE_IDX]
sample_entry = samples[sample_oo3["id"]]

print(f"Sample {SAMPLE_IDX}  id={sample_oo3['id']}")
print(f"Q: {sample_oo3['question']}")
print(f"A: {sample_oo3['answer']}")
print(f"Concepts: {[c['concept'] for c in sample_entry['concepts']]}")

## OO3-3. Build annotated frame stack (1fps + spliced keyframes + bbox overlays)

In [ ]:
# reconstruct augmented frame stack (same as sam3_track.py)
frames = fetch_video({"video": get_video_path(sample_oo3), "fps": VIDEO_FPS})
frames_list = list(frames.permute(0, 2, 3, 1).cpu().numpy().astype("uint8"))
H, W = frames_list[0].shape[:2]
splice_keyframes(sample_oo3, frames_list, H, W)
frames_aug = np.stack(frames_list)  # (T, H, W, 3)
T = len(frames_aug)
print(f"Augmented frame stack: {T} frames, {W}x{H}")

annotated = frames_aug.copy()

if ANNOTATE_BBOX:
    masks_dict = load_track_masks(sample_entry, RESULTS_DIR)
    concepts = list(masks_dict.keys())
    print(f"{len(concepts)} track(s) available: {concepts}")

    # overlay every concept's bbox track onto the frame stack (all red)
    for concept in concepts:
        bboxes = masks_to_bboxes(masks_dict[concept], target_size=(H, W))
        for t, (x1, y1, x2, y2) in enumerate(bboxes.astype(int)):
            if x2 > x1 and y2 > y1:
                cv2.rectangle(annotated[t], (x1, y1), (x2, y2), color=(255, 0, 0), thickness=2)

# subsample to TGT_FRAMES
indices = np.linspace(0, T - 1, TGT_FRAMES, dtype=int)
annotated_input_oo3 = torch.tensor(annotated[indices]).permute(0, 3, 1, 2).to(torch.float32)
print(f"annotated_input: {annotated_input_oo3.shape}")

## OO3-4. Preview a random annotated frame

In [ ]:
import matplotlib.pyplot as plt

random_idx = np.random.randint(0, annotated_input_oo3.shape[0])
any_frame = annotated_input_oo3[random_idx].permute(1, 2, 0).numpy().astype(np.uint8)
plt.imshow(any_frame)
plt.title(f"Annotated frame {random_idx} — id: {sample_oo3['id']}")
plt.axis('off')
plt.show()

## OO3-5. Save annotated video to disk

In [ ]:
from torchcodec.encoders import VideoEncoder

dump_dir = "/projects/torresani-lab/ajay/visual-grounded-vlm/scripts/vllm_serving/dump"
os.makedirs(dump_dir, exist_ok=True)
output_video_path_oo3 = os.path.join(dump_dir, f"oo3_ann_{sample_oo3['id']}.mp4")
VideoEncoder(annotated_input_oo3.to(torch.uint8), frame_rate=4).to_file(output_video_path_oo3)
print("wrote:", output_video_path_oo3)

## OO3-6. Build the chat request and send to vLLM

In [ ]:
# def create_user_prompt(video_summary, question, concept_captions, answer=None):
#     prompt = f"Video summary: {video_summary}\n"
#     prompt += f"Question: {question}\n"
#     prompt += "Relevant concepts and their captions:\n"
#     for concept, caption in concept_captions.items():
#         prompt += f"- {concept}: {caption}\n"
#     if answer is not None:
#         prompt += f"Answer: {answer}\n"
#     return prompt
def create_user_prompt(video_summary, question, concept_captions, answer=None):
    prompt = f"{video_summary}\n"
    prompt += "Red bounding box annotated objects/entities in the video:\n"
    for concept, caption in concept_captions.items():
        prompt += f"- {concept}: {caption}\n"
    prompt += f"Question: {question}\n"
    if answer is not None:
        prompt += f"Ground Truth Answer: {answer}\n"
    return prompt

# load the PLM rcap outputs (video summary + per-concept captions) for the current sample
RCAP_DIR = "/scratch/rajendrakumar.a/visual-grounded-vlm/annotations/openo3video/openo3_str_plm_rcap"
rcap = json.load(open(os.path.join(RCAP_DIR, "samples", f"{sample_oo3['id']}.json")))

user_prompt = create_user_prompt(
    video_summary=rcap["video_summary"],
    question=sample_oo3["question"],
    concept_captions=rcap["concept_captions"],
    answer=sample_oo3["answer"],
)
print(user_prompt)

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
model = "Qwen/Qwen3-VL-235B-A22B-Instruct"

SYSTEM_PROMPT_PATH = "/projects/torresani-lab/ajay/visual-grounded-vlm/prompts/cot/v8.md"
with open(SYSTEM_PROMPT_PATH, "r") as f:
    SYSTEM_PROMPT = f.read()

n_frames = annotated_input_oo3.shape[0]
frame_prompt = "".join(f"Frame {i}: <image>\n" for i in range(n_frames))

message_oo3 = [
    {
        "role": "system",
        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": frame_prompt + user_prompt},
            {
                "type": "video_url",
                "video_url": {"url": "file://" + output_video_path_oo3},
                # "VIDEO_MIN_PIXELS": 512 * 28 * 28,
                "VIDEO_MAX_PIXELS": 1024 * 28 * 28,
                "fps": 1,
            },
        ],
    },
]

client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)

In [ ]:
chat_response = client.chat.completions.create(
    model=model,
    messages=message_oo3,
    max_tokens=768,
    seed=42,
    temperature=0,
)
response_text = chat_response.choices[0].message.content
print(response_text)

In [ ]:
print(f"Question: {sample_oo3['question']}")
print(f"Ground truth answer: {sample_oo3['answer']}")
print(f"CoT: {sample_oo3['reasoning_process']}")

print(f"Sample {SAMPLE_IDX}  id={sample_oo3['id']}")
print(f"Q: {sample_oo3['question']}")
print(f"A: {sample_oo3['answer']}")
print(f"CoT: {sample_oo3['reasoning_process']}")
print(f"Concepts: {[c['concept'] for c in sample_entry['concepts']]}")

---
# Section 3: Open-o3-Video Region Captioning (single annotated entity)

Pick one tracked concept, overlay only its bbox across frames, and ask the model to caption what is happening to that highlighted entity. Reuses the dataset, track manifest, and helpers loaded in Section 2.

In [ ]:
import sys
sys.path.append("/projects/torresani-lab/ajay/visual-grounded-vlm/")

import importlib
import json
import os

import cv2
import numpy as np
import torch
from datasets import load_dataset
from qwen_vl_utils import fetch_video

import src.utils.openo3_utils as openo3_utils
importlib.reload(openo3_utils)
from src.utils.openo3_utils import (
    get_video_path, splice_keyframes,
    load_track_manifest, load_track_masks, masks_to_bboxes,
)

# --- config ---
OO3_JSON    = "/projects/torresani-lab/ajay/datasets/Open-o3-Video/json_data/STGR-SFT.json"
RESULTS_DIR = "/scratch/rajendrakumar.a/visual-grounded-vlm/annotations/openo3video/uniform_sample_1fps_orig_res_track"

dataset_oo3 = load_dataset("json", data_files=OO3_JSON, split="train")
str_dataset = [s for s in dataset_oo3 if s["task"] == "temporal-spatial free-form QA"]
run_meta, samples = load_track_manifest(RESULTS_DIR)

print(f"Loaded {len(str_dataset)} STGR samples")
print(f"Track manifest: {len(samples)} entries")

## RCAP-1. Pick a sample and one concept to annotate

In [ ]:
# --- config ---
RCAP_SAMPLE_IDX = 15
CONCEPT_IDX     = 0      # which tracked concept to highlight
VIDEO_FPS       = run_meta["config"]["video"]["fps"]
TGT_FRAMES      = 32

rcap_sample = str_dataset[RCAP_SAMPLE_IDX]
rcap_entry  = samples[rcap_sample["id"]]

# load this sample's tracks and pick the single concept to annotate
masks_dict_rcap = load_track_masks(rcap_entry, RESULTS_DIR)
concept = list(masks_dict_rcap.keys())[CONCEPT_IDX]

print(f"Sample {RCAP_SAMPLE_IDX}  id={rcap_sample['id']}")
print(f"Available concepts: {list(masks_dict_rcap.keys())}")
print(f"Captioning concept: {concept!r}")

## RCAP-2. Build the frame stack with only that concept's bbox overlaid

In [ ]:
# reconstruct augmented frame stack (1fps + spliced keyframes), same as Section 2
frames = fetch_video({"video": get_video_path(rcap_sample), "fps": VIDEO_FPS})
frames_list = list(frames.permute(0, 2, 3, 1).cpu().numpy().astype("uint8"))
H, W = frames_list[0].shape[:2]
splice_keyframes(rcap_sample, frames_list, H, W)
frames_aug = np.stack(frames_list)  # (T, H, W, 3)
T = len(frames_aug)

# overlay only the chosen concept's bbox track (red)
annotated = frames_aug.copy()
bboxes = masks_to_bboxes(masks_dict_rcap[concept], target_size=(H, W))
for t, (x1, y1, x2, y2) in enumerate(bboxes.astype(int)):
    if x2 > x1 and y2 > y1:
        cv2.rectangle(annotated[t], (x1, y1), (x2, y2), color=(255, 0, 0), thickness=2)

# subsample to TGT_FRAMES
indices = np.linspace(0, T - 1, TGT_FRAMES, dtype=int)
annotated_input_rcap = torch.tensor(annotated[indices]).permute(0, 3, 1, 2).to(torch.float32)
print(f"annotated_input: {annotated_input_rcap.shape}")

## RCAP-3. Save annotated video to disk

In [ ]:
from torchcodec.encoders import VideoEncoder

dump_dir = "/projects/torresani-lab/ajay/visual-grounded-vlm/scripts/vllm_serving/dump"
os.makedirs(dump_dir, exist_ok=True)
concept_slug = concept.replace(" ", "_")
output_video_path_rcap = os.path.join(dump_dir, f"rcap_{rcap_sample['id']}_{concept_slug}.mp4")
VideoEncoder(annotated_input_rcap.to(torch.uint8), frame_rate=4).to_file(output_video_path_rcap)
print("wrote:", output_video_path_rcap)

## RCAP-4. Build the chat request and send to vLLM

In [ ]:
from openai import OpenAI

openai_api_key = "EMPTY"
openai_api_base = "http://localhost:8000/v1"
model = "Qwen/Qwen3-VL-235B-A22B-Instruct"

SYSTEM_PROMPT_PATH = "/projects/torresani-lab/ajay/visual-grounded-vlm/prompts/rcap/v2.md"
with open(SYSTEM_PROMPT_PATH, "r") as f:
    rcap_prompt = f.read()
    
n_frames = annotated_input_rcap.shape[0]
# frame_prompt = "".join(f"Frame {i}: <image>\n" for i in range(n_frames))

message_rcap = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": rcap_prompt},
            {
                "type": "video_url",
                "video_url": {"url": "file://" + output_video_path_rcap},
                "VIDEO_MAX_PIXELS": 1024 * 28 * 28,
                "fps": 1,
            },
        ],
    },
]

client = OpenAI(api_key=openai_api_key, base_url=openai_api_base)

In [ ]:
chat_response = client.chat.completions.create(
    model=model,
    messages=message_rcap,
    max_tokens=768,
    seed=42,
    temperature=0,
)
response_text = chat_response.choices[0].message.content
print(f"Concept: {concept}\n")
print(response_text)

In [ ]:
# For comparison: PLM's caption for the same concept
RCAP_DIR = "/scratch/rajendrakumar.a/visual-grounded-vlm/annotations/openo3video/openo3_str_plm_rcap"
plm_rcap = json.load(open(os.path.join(RCAP_DIR, "samples", f"{rcap_sample['id']}.json")))

print(f"Concept: {concept}\n")
print(plm_rcap["concept_captions"][concept])